In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, sys
pkgs = ["torch", "torchvision", "timm", "scikit-learn", "openpyxl",
        "pandas", "Pillow", "numpy", "tqdm", "matplotlib"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)


In [ ]:
import os, csv, json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
import timm
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
%matplotlib inline

# Try importing TPU (torch_xla)
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    TPU_AVAILABLE = True
except:
    TPU_AVAILABLE = False

# Device selection logic
if TPU_AVAILABLE:
    device = xm.xla_device()
    device_name = "TPU"
elif torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"GPU – {torch.cuda.get_device_name(0)}"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device_name}")

In [ ]:
from pathlib import Path
import torch
import os

class Config:

    # ── Base folder in Google Drive ────────────────────────────────────────
    BASE_PATH = Path("/content/drive/MyDrive/fundusproject")

    # ── File paths ─────────────────────────────────────────────────────────
    EXCEL_PATH = BASE_PATH / "VF_and_clinical_information.xlsx"
    IMAGE_DIR = BASE_PATH / "Annotated images"

    CHECKPOINT_DIR = BASE_PATH / "checkpoints"
    RESULTS_DIR = BASE_PATH / "test_results"

    SHEET_NAME = "Baseline"

    # ── Excel column indices (0-based, after pandas reads header row 1) ────
    COL_AGE = 3        # C – Age
    COL_GENDER = 4     # D – Gender
    COL_IOP = 5        # E – IOP
    COL_CCT = 6        # F – CCT
    COL_CFP = 17       # Q – Fundus image filename

    # Cols G–K (6–10) and R–S (17–18) are intentionally ignored
    COL_VF_START = 20  # T – first VF point
    COL_VF_END = 81    # CB – last VF point → 61 values

    # ── Model dimensions ────────────────────────────────────────────────────
    VIT_MODEL = "vit_base_patch16_224"
    VIT_EMBED_DIM = 768

    CLINICAL_DIM = 4
    MLP_HIDDEN = 128
    MLP_EMBED_DIM = 128

    COND_DIM = VIT_EMBED_DIM + MLP_EMBED_DIM
    VF_DIM = 61

    # ── Diffusion schedule ──────────────────────────────────────────────────
    T_STEPS = 1000
    BETA_START = 5e-5
    BETA_END = 0.01

    # ── Training ────────────────────────────────────────────────────────────
    BATCH_SIZE = 8
    EPOCHS = 200
    LR = 1e-4
    TRAIN_RATIO = 0.70
    RANDOM_SEED = 42

    IMAGE_SIZE = 224
    NUM_WORKERS = 2

    LAMBDA_CONTRAST = 0.1

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


cfg = Config()

# Create folders in Google Drive
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)

print("Config ready. Device:", cfg.DEVICE)

In [ ]:
class FundusViTEncoder(nn.Module):
    """
    timm ViT-B/16.  Returns the [CLS] token embedding (768-d).
    Optional: pass fundus_ckpt='RETFound_cfp_weights.pth' for retinal weights.
    Download from: https://github.com/rmaphoh/RETFound_MAE
    """
    def __init__(self, model_name=cfg.VIT_MODEL, pretrained=True, fundus_ckpt=None):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        if fundus_ckpt and os.path.exists(fundus_ckpt):
            state = torch.load(fundus_ckpt, map_location="cpu")
            key   = "model" if "model" in state else "state_dict"
            self.vit.load_state_dict(state.get(key, state), strict=False)
            print(f"[ViT] Loaded fundus checkpoint: {fundus_ckpt}")

    def forward(self, x):
        return self.vit(x)   # (B, 768)

In [ ]:
class ClinicalMLP(nn.Module):
    def __init__(self, in_dim=cfg.CLINICAL_DIM, hidden=cfg.MLP_HIDDEN,
                 out_dim=cfg.MLP_EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)   # (B, 128)

In [ ]:
class ConditioningFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(cfg.VIT_EMBED_DIM + cfg.MLP_EMBED_DIM, cfg.COND_DIM),
            nn.LayerNorm(cfg.COND_DIM),
            nn.GELU(),
        )
    def forward(self, vit_emb, mlp_emb):
        return self.proj(torch.cat([vit_emb, mlp_emb], dim=-1))   # (B, 896)


In [ ]:
class DiffusionSchedule:
    def __init__(self, T=cfg.T_STEPS, beta_start=cfg.BETA_START, beta_end=cfg.BETA_END):
        self.T         = T
        betas          = torch.linspace(beta_start, beta_end, T)
        alphas         = 1.0 - betas
        alpha_bar      = torch.cumprod(alphas, dim=0)
        self.betas     = betas
        self.alphas    = alphas
        self.alpha_bar = alpha_bar
        self.sqrt_ab   = torch.sqrt(alpha_bar)
        self.sqrt_1mab = torch.sqrt(1.0 - alpha_bar)

    def to(self, device):
        for attr in ("betas", "alphas", "alpha_bar", "sqrt_ab", "sqrt_1mab"):
            setattr(self, attr, getattr(self, attr).to(device))
        return self

    def q_sample(self, x0, t, noise=None):
        """Forward diffusion q(x_t|x_0): add noise at step t."""
        if noise is None:
            noise = torch.randn_like(x0)
        s  = self.sqrt_ab[t].unsqueeze(-1)
        sm = self.sqrt_1mab[t].unsqueeze(-1)
        return s * x0 + sm * noise, noise

In [ ]:
class SinusoidalTimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freq = torch.exp(-np.log(10000) *
                         torch.arange(half, device=t.device) / (half - 1))
        args = t.float().unsqueeze(-1) * freq.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)


class ResBlock1D(nn.Module):
    """Residual block conditioned on time + conditioning vector."""
    def __init__(self, channels, time_dim, cond_dim):
        super().__init__()
        self.norm1     = nn.LayerNorm(channels)
        self.fc1       = nn.Linear(channels, channels)
        self.norm2     = nn.LayerNorm(channels)
        self.fc2       = nn.Linear(channels, channels)
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, channels))
        self.cond_proj = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, channels))

    def forward(self, x, t_emb, cond):
        h = self.fc1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb) + self.cond_proj(cond)
        h = self.fc2(F.silu(self.norm2(h)))
        return x + h


class VFDenoiser(nn.Module):
    """
    Predicts noise ε at step t.
    Input  : (B,61) x_t  +  (B,) t  +  (B,896) cond
    Output : (B,61) predicted noise
    """
    def __init__(self, vf_dim=cfg.VF_DIM, cond_dim=cfg.COND_DIM,
                 hidden=256, depth=6, time_dim=128):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbed(time_dim),
            nn.Linear(time_dim, time_dim * 4), nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )
        self.input_proj  = nn.Linear(vf_dim, hidden)
        self.blocks      = nn.ModuleList(
            [ResBlock1D(hidden, time_dim, cond_dim) for _ in range(depth)])
        self.output_proj = nn.Sequential(
            nn.LayerNorm(hidden), nn.Linear(hidden, vf_dim))

    def forward(self, x_t, t, cond):
        t_emb = self.time_embed(t)
        h     = self.input_proj(x_t)
        for block in self.blocks:
            h = block(h, t_emb, cond)
        return self.output_proj(h)


In [ ]:
class CDPMVFSynthesizer(nn.Module):
    def __init__(self, fundus_ckpt=None):
        super().__init__()
        self.vit      = FundusViTEncoder(fundus_ckpt=fundus_ckpt)
        self.mlp      = ClinicalMLP()
        self.fusion   = ConditioningFusion()
        self.denoiser = VFDenoiser()

    def encode(self, images, clinical):
        """Fuse fundus image + clinical embeddings → conditioning vector (B,896)."""
        return self.fusion(self.vit(images), self.mlp(clinical))

    def forward(self, images, clinical, x_t, t):
        """Predict noise ε added at diffusion step t."""
        return self.denoiser(x_t, t, self.encode(images, clinical))

# Quick parameter count
_m = CDPMVFSynthesizer()
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
del _m


In [ ]:
checkpoint = torch.load(
    "/content/drive/MyDrive/fundusproject/checkpoints/best_model.pt",
    map_location="cpu",
    weights_only=False   #
)

In [ ]:
model = CDPMVFSynthesizer()

In [ ]:
model.load_state_dict(checkpoint["model_state"])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
model.eval()

In [ ]:
@torch.no_grad()
def ddpm_reverse_sample(model, schedule, cond, device, n_steps=300):
    """
    Full DDPM reverse chain: x_T ~ N(0,I)  →  x_0  (synthesised VF, z-score space).

    Parameters
    ----------
    cond    : (B, COND_DIM)  conditioning vector from model.encode()
    Returns : (B, VF_DIM)   synthesised VF, still z-scored
    """
    x = torch.randn(cond.size(0), cfg.VF_DIM, device=device)

    for t_idx in tqdm(reversed(range(n_steps)),
                      desc="DDPM reverse sampling", total=n_steps, leave=False):
        t_tensor = torch.full((cond.size(0),), t_idx, device=device, dtype=torch.long)
        pred_eps = model.denoiser(x, t_tensor, cond)

        beta_t         = schedule.betas[t_idx]
        alpha_t        = schedule.alphas[t_idx]
        alpha_bar_t    = schedule.alpha_bar[t_idx]
        alpha_bar_prev = (schedule.alpha_bar[t_idx - 1] if t_idx > 0
                          else torch.tensor(1.0, device=device))

        # Estimate clean x_0
        x0_pred = (x - torch.sqrt(1.0 - alpha_bar_t) * pred_eps) / torch.sqrt(alpha_bar_t)
        x0_pred = x0_pred.clamp(-3.0, 3.0)

        # Posterior mean
        coef1 = torch.sqrt(alpha_bar_prev) * beta_t / (1.0 - alpha_bar_t)
        coef2 = torch.sqrt(alpha_t) * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)
        mean  = coef1 * x0_pred + coef2 * x

        if t_idx > 0:
            var = beta_t * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)
            x   = mean + torch.sqrt(var) * torch.randn_like(x)
        else:
            x = mean

    return x


In [ ]:
INFER_IMAGE  = str(cfg.IMAGE_DIR / "1_OD_1.jpg")
INFER_AGE    = 46.0
INFER_GENDER = "F"
INFER_IOP    = 14.7
INFER_CCT    = 535.0

In [ ]:
def infer_single(image_path, age, gender, iop, cct, ckpt_path=None):
    print(" Function started")

    device = torch.device(cfg.DEVICE)
    ckpt_path = ckpt_path or str(Path(cfg.CHECKPOINT_DIR) / "best_model.pt")

    print(" Loading checkpoint...")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    print(" Loading model...")
    model = CDPMVFSynthesizer().to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    print(" Loading schedule...")
    schedule = DiffusionSchedule().to(device)

    #  OPTIONAL: speed up for testing
    if hasattr(schedule, "T"):
        schedule.T = 50   # reduce from 1000 → FAST

    vf_mean   = np.array(ckpt["vf_mean"], dtype=np.float32)
    vf_std    = np.array(ckpt["vf_std"], dtype=np.float32)
    clin_mean = np.array(ckpt["clin_mean"], dtype=np.float32)
    clin_std  = np.array(ckpt["clin_std"], dtype=np.float32)

    print(" Loading image...")
    tfm = transforms.Compose([
        transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])

    img = Image.open(image_path).convert("RGB")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].set_title("Input Fundus Image")
    axes[0].axis("off")

    img_t = tfm(img).unsqueeze(0).to(device)

    print(" Processing clinical data...")
    gender_val = 1.0 if str(gender).upper().startswith("F") else 0.0
    clin_raw   = np.array([age, gender_val, iop, cct], dtype=np.float32)

    clin_t = torch.tensor(
        (clin_raw - clin_mean) / clin_std,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    print(" Running diffusion...")
    with torch.no_grad():
        cond = model.encode(img_t, clin_t)
        synth_norm = ddpm_reverse_sample(model, schedule, cond, device)

    print("Inference done")

    synth_vf = synth_norm.cpu().numpy().squeeze() * vf_std + vf_mean

    print(" Output shape:", synth_vf.shape)
    print("\nSynthesised Visual Field (61 points):")
    for i, v in enumerate(synth_vf):
        print(f"VF[{i+1:02d}] = {v:.3f}")

    # Plot
    axes[1].bar(np.arange(1, cfg.VF_DIM + 1), synth_vf)
    axes[1].set_title("Synthesised VF")
    plt.tight_layout()
    plt.show()

    return synth_vf

In [ ]:
    synth_vf = infer_single(
    INFER_IMAGE,
    INFER_AGE,
    INFER_GENDER,
    INFER_IOP,
    INFER_CCT
    )

In [ ]:
def vf_to_grid(vf):
    grid = np.full((10, 10), np.nan)

    # Standard approximate mapping (Humphrey 24-2 style)
    indices = [
        (0,3),(0,4),(0,5),(0,6),
        (1,2),(1,3),(1,4),(1,5),(1,6),(1,7),
        (2,1),(2,2),(2,3),(2,4),(2,5),(2,6),(2,7),(2,8),
        (3,1),(3,2),(3,3),(3,4),(3,5),(3,6),(3,7),(3,8),
        (4,1),(4,2),(4,3),(4,4),(4,5),(4,6),(4,7),(4,8),
        (5,1),(5,2),(5,3),(5,4),(5,5),(5,6),(5,7),(5,8),
        (6,1),(6,2),(6,3),(6,4),(6,5),(6,6),(6,7),(6,8),
        (7,2),(7,3),(7,4),(7,5),(7,6),(7,7),
        (8,3),(8,4),(8,5),(8,6)
    ]

    for i, (r,c) in enumerate(indices):
        if i < len(vf):
            grid[r,c] = vf[i]

    return grid

In [ ]:
import matplotlib.pyplot as plt

def plot_vf_heatmap(vf):
    grid = vf_to_grid(vf)

    plt.figure(figsize=(6,6))
    plt.imshow(grid, cmap='jet', interpolation='nearest')
    plt.colorbar(label="Sensitivity (dB)")
    plt.title("Visual Field Heatmap")
    plt.axis("off")
    plt.show()

In [ ]:
plot_vf_heatmap(synth_vf)